# ClaimGuard research findings

This notebook reconstructs the evidence behind all twelve research findings from persisted held-out results. It does not retrain models.


## Load canonical evidence

Run the following cell after generating the model reports. It supports Jupyter launched from either the repository root or the notebooks directory.


In [8]:
from pathlib import Path

import json
import pandas as pd

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'reports').is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Could not locate the ClaimGuard reports directory')
reports = project_root / 'reports'

def load_json(directory, filename='metrics.json'):
    return json.loads((reports / directory / filename).read_text())

frequency = pd.read_csv(reports / 'ml_frequency' / 'model_comparison.csv')
severity = pd.read_csv(reports / 'ml_severity' / 'model_comparison.csv')
premium = pd.read_csv(reports / 'ml_pure_premium' / 'model_comparison.csv')
benchmark = pd.read_csv(reports / 'model_benchmark' / 'model_benchmark.csv')
tail = pd.read_csv(reports / 'tail_performance' / 'tail_performance.csv')
frequency_importance = pd.read_csv(reports / 'explainability' / 'frequency_permutation_importance.csv')
severity_importance = pd.read_csv(reports / 'explainability' / 'severity_permutation_importance.csv')
feature_ranks = pd.read_csv(reports / 'explainability' / 'feature_rank_comparison.csv')
bonus = load_json('bonus_malus')
calibration = load_json('calibration')
deciles = load_json('risk_deciles')
large_loss = load_json('large_loss_classification')
stress = load_json('portfolio_stress')
evt = load_json('extreme_value')
rigor = load_json('statistical_rigor')
print(f'Loaded ClaimGuard evidence from {reports}')


Loaded ClaimGuard evidence from /Users/hkbuttar/claimguard/reports


## 1. Which characteristics are associated with claim frequency?

**Robust** — Bonus-Malus and vehicle age ranked first and second across GLM, permutation, and SHAP frequency explanations; driver age was also consistently important.

Expected evidence: Permutation top five: BonusMalus, VehAge, VehBrand, VehPower, DrivAge.


In [9]:
frequency_ranks = feature_ranks[['Feature', 'FrequencyGLMRank', 'FrequencyPermutationRank', 'FrequencySHAPRank']]
display(frequency_ranks.sort_values('FrequencyPermutationRank'))


,Feature,FrequencyGLMRank,FrequencyPermutationRank,FrequencySHAPRank
4,BonusMalus,1.0,1.0,1.0
2,VehAge,2.0,2.0,2.0
5,VehBrand,4.0,3.0,7.0
1,VehPower,5.0,4.0,6.0
3,DrivAge,3.0,5.0,3.0
6,VehGas,7.0,6.0,8.0
8,Region,6.0,7.0,5.0
7,Density,8.0,8.0,4.0
0,Area,9.0,9.0,9.0


## 2. Which characteristics are associated with claim severity?

**Suggestive** — Severity signals were weaker and less consistent; region, Bonus-Malus, driver age, and vehicle power appeared important under different explanation methods.

Expected evidence: SHAP top five: DrivAge, Region, VehPower, VehBrand, BonusMalus.


In [10]:
severity_ranks = feature_ranks[['Feature', 'SeverityGLMRank', 'SeverityPermutationRank', 'SeveritySHAPRank']]
display(severity_ranks.sort_values('SeveritySHAPRank'))


,Feature,SeverityGLMRank,SeverityPermutationRank,SeveritySHAPRank
3,DrivAge,6.0,8.0,1.0
8,Region,4.0,2.0,2.0
1,VehPower,1.0,5.0,3.0
5,VehBrand,3.0,7.0,4.0
4,BonusMalus,2.0,1.0,5.0
2,VehAge,9.0,9.0,6.0
7,Density,7.0,6.0,7.0
0,Area,5.0,3.0,8.0
6,VehGas,8.0,4.0,9.0


## 3. How different are the drivers of frequency and severity?

**Suggestive** — Frequency is dominated by Bonus-Malus and vehicle age, whereas severity gives more weight to driver age, region, and vehicle power.

Expected evidence: Only Bonus-Malus appears in every frequency and severity top-five comparison.


In [11]:
display(feature_ranks.set_index('Feature').sort_index())


,FrequencyGLMRank,FrequencyPermutationRank,FrequencySHAPRank,SeverityGLMRank,SeverityPermutationRank,SeveritySHAPRank
Feature,,,,,,
Area,9.0,9.0,9.0,5.0,3.0,8.0
BonusMalus,1.0,1.0,1.0,2.0,1.0,5.0
Density,8.0,8.0,4.0,7.0,6.0,7.0
DrivAge,3.0,5.0,3.0,6.0,8.0,1.0
Region,6.0,7.0,5.0,4.0,2.0,2.0
VehAge,2.0,2.0,2.0,9.0,9.0,6.0
VehBrand,4.0,3.0,7.0,3.0,7.0,4.0
VehGas,7.0,6.0,8.0,8.0,4.0,9.0
VehPower,5.0,4.0,6.0,1.0,5.0,3.0


## 4. How well does Bonus-Malus separate actual risk?

**Robust** — Bonus-Malus separates frequency and pure premium after controlling for driver, vehicle, and geography, but adds no held-out severity value.

Expected evidence: Per +10 points: frequency relativity 1.252, pure-premium relativity 1.470; severity deviance change -0.13%.


In [12]:
display(pd.DataFrame({task: values for task, values in bonus.items() if task in ('frequency', 'severity', 'pure_premium')}).T[['relativity_per_10_points', 'relativity_95_low', 'relativity_95_high', 'deviance_improvement']])


,relativity_per_10_points,relativity_95_low,relativity_95_high,deviance_improvement
frequency,1.251715,1.243348,1.260137,0.018649
severity,1.073529,1.025115,1.124229,-0.001281
pure_premium,1.470238,1.348591,1.602859,0.014493


## 5. Do ML models materially outperform actuarial GLMs?

**Robust** — ML materially improves some predictive metrics, but not calibration, distribution fit, tail protection, or interpretability; no universal winner exists.

Expected evidence: Frequency deviance improvement 5.09%; severity MAE improvement 17.24%.


In [13]:
display(benchmark[['Task', 'Metric', 'TraditionalModel', 'TraditionalValue', 'MLModel', 'MLValue', 'Winner']])


,Task,Metric,TraditionalModel,TraditionalValue,MLModel,MLValue,Winner
0,Frequency,Mean Poisson deviance,Poisson GLM,0.321152,HistGradientBoosting,0.304806,ML
1,Severity average error,MAE (€),Gamma GLM,2147.693300,XGBoost,1777.387551,ML
2,Severity distribution,Mean Gamma deviance,Lognormal,1.632402,XGBoost,1.726474,Traditional
3,Pure premium,Mean Tweedie deviance,Tweedie GLM,70.666601,GBM components,68.261457,ML
4,Risk ranking,Normalized Gini,Tweedie GLM,0.188895,Direct Boosting,0.229411,ML
5,Pure-premium calibration,Absolute aggregate ratio error,Tweedie GLM,0.006683,GBM components,0.161282,Traditional
6,Top-5% severity,MAE (€),Lognormal,19278.672874,Random Forest,19833.068919,Traditional
7,Top-1% severity capture,Predicted/observed loss,Lognormal,0.027760,Random Forest,0.028930,Neither
8,Interpretability,Transparent coefficients and uncertainty,GLMs,1.000000,Boosting + SHAP,0.000000,Traditional


## 6. Is ML improvement present for frequency, severity, or both?

**Robust** — Improvements are present for frequency deviance and severity MAE, with paired bootstrap support for both.

Expected evidence: Stable frequency improvement: True; stable severity improvement: True.


In [14]:
display(pd.DataFrame(rigor['comparisons'])[['Comparison', 'TraditionalModel', 'MLModel', 'lower', 'upper', 'StatisticallyStableMLImprovement']])


,Comparison,TraditionalModel,MLModel,lower,upper,StatisticallyStableMLImprovement
0,Frequency deviance,Poisson GLM,HistGradientBoosting,-0.018205,-0.014598,True
1,Severity MAE,Gamma GLM,XGBoost,-396.301661,-347.514004,True
2,Pure-premium deviance,Tweedie GLM,GBM Components,-4.915270,-0.242327,True


## 7. Does frequency × severity outperform direct Tweedie modeling?

**Suggestive** — Component GBM has lower Tweedie deviance, while direct Tweedie has substantially better aggregate calibration and lower MAE than traditional component models.

Expected evidence: Component GBM deviance 68.261; Tweedie GLM 70.667.


In [15]:
display(premium[['Model', 'mean_tweedie_deviance', 'mae', 'predicted_observed_ratio', 'normalized_gini']].sort_values('mean_tweedie_deviance'))


,Model,mean_tweedie_deviance,mae,predicted_observed_ratio,normalized_gini
3,GBM Frequency × GBM Severity,68.261457,176.297647,1.161282,0.223210
1,Poisson × Lognormal,68.670467,198.041746,1.421727,0.141936
0,Poisson × Gamma,70.001863,190.705480,1.332631,0.112385
2,Tweedie GLM,70.666601,162.948190,0.993317,0.188895
4,Direct Boosting,77.046198,138.848294,0.686004,0.229411


## 8. Which approach produces the best calibrated pure premium?

**Robust** — The Tweedie GLM is the best aggregate-calibrated held-out pure-premium model.

Expected evidence: Tweedie predicted/observed ratio: 0.9933.


In [16]:
pure_premium_calibration = pd.DataFrame(calibration['pure_premium']).T
pure_premium_calibration['absolute_ratio_error'] = (pure_premium_calibration['observed_expected_ratio'] - 1).abs()
display(pure_premium_calibration.sort_values('absolute_ratio_error'))


,observed,expected,observed_expected_ratio,aggregate_bias,mean_bias,segment_weighted_absolute_loss_cost_error,absolute_ratio_error
Tweedie GLM,11868103.84,1.178879e+07,1.006728,-7.931184e+04,-0.584883,31.405984,0.006728
GBM Components,11868103.84,1.378221e+07,0.861118,1.914106e+06,14.115515,27.640045,0.138882
Poisson × Gamma,11868103.84,1.581580e+07,0.750396,3.947693e+06,29.112139,55.138314,0.249604
Poisson × Lognormal,11868103.84,1.687320e+07,0.703370,5.005098e+06,36.909938,69.907325,0.296630
Direct Boosting,11868103.84,8.141564e+06,1.457718,-3.726540e+06,-27.481249,52.049413,0.457718


## 9. Do average-performance winners remain strong in the extreme tail?

**Robust** — No. Every severity model misses more than 97% of held-out top-1% aggregate loss.

Expected evidence: Best top-1% capture was 2.89% from Random Forest.


In [17]:
top_one = tail.loc[tail['Segment'].eq('Top 1%'), ['Model', 'claims', 'mae', 'observed_loss', 'predicted_loss', 'predicted_observed_ratio']]
display(top_one.sort_values('predicted_observed_ratio', ascending=False))


,Model,claims,mae,observed_loss,predicted_loss,predicted_observed_ratio
27,Random Forest,45,85183.391717,3947454.37,114201.742736,0.028930
26,Lognormal,45,85286.090470,3947454.37,109580.298866,0.027760
24,Mean,45,85426.961710,3947454.37,103241.093030,0.026154
25,Gamma GLM,45,85505.208999,3947454.37,99719.965029,0.025262
28,HistGradientBoosting,45,85756.943308,3947454.37,88391.921150,0.022392
29,XGBoost,45,85888.091005,3947454.37,82490.274780,0.020897


## 10. How concentrated is portfolio risk among high-risk policies?

**Suggestive** — Model-ranked high-risk policies concentrate loss, but rankings are noisy because isolated extreme claims dominate observed outcomes.

Expected evidence: Direct boosting D10 captured 28.91%; GBM components D10 captured 23.88%.


In [18]:
decile_evidence = pd.DataFrame(deciles['diagnostics']).T
display(decile_evidence[['spearman_decile_vs_observed_loss_cost', 'highest_to_lowest_loss_cost_ratio', 'highest_decile_loss_capture', 'aggregate_observed_expected_ratio']].sort_values('highest_decile_loss_capture', ascending=False))


,spearman_decile_vs_observed_loss_cost,highest_to_lowest_loss_cost_ratio,highest_decile_loss_capture,aggregate_observed_expected_ratio
direct_boosting,0.672727,3.674826,0.289064,1.457718
gbm_component,0.733333,8.330043,0.238817,0.861118
tweedie,0.684848,3.084805,0.181518,1.006728
poisson_gamma,0.721212,3.692416,0.138844,0.750396
poisson_lognormal,0.866667,4.841926,0.130227,0.70337


## 11. Can large claims be identified before they occur?

**Data-limited** — Available policy characteristics provide only weak discrimination for large claims and essentially none for the most extreme claims.

Expected evidence: Q95 boosting PR-AUC 0.0628 at event rate 4.92%.


In [19]:
rows = []
for threshold, details in large_loss['thresholds'].items():
    for model, metrics in details['models'].items():
        rows.append({'Threshold': threshold, 'ClaimAmount': details['claim_amount_threshold'], 'Model': model, **metrics})
display(pd.DataFrame(rows)[['Threshold', 'ClaimAmount', 'Model', 'event_rate', 'pr_auc', 'roc_auc', 'recall', 'precision']])


,Threshold,ClaimAmount,Model,event_rate,pr_auc,roc_auc,recall,precision
0,q90,2779.0200,logistic,0.097968,0.120065,0.559540,0.474806,0.114113
1,q90,2779.0200,gradient_boosting,0.097968,0.118991,0.566999,0.337209,0.124821
2,q95,4808.0080,logistic,0.049174,0.059446,0.561903,0.247104,0.066667
3,q95,4808.0080,gradient_boosting,0.049174,0.062822,0.574669,0.092664,0.067227
4,q99,17083.1236,logistic,0.008544,0.008154,0.476586,0.022222,0.002532
5,q99,17083.1236,gradient_boosting,0.008544,0.007828,0.465765,0.000000,0.000000


## 12. How much aggregate loss uncertainty comes from extreme claims?

**Exploratory** — EVT stress scenarios are overwhelmingly tail-driven, but the fitted infinite-variance tail makes capital metrics highly unstable.

Expected evidence: Tail share of mean loss 62.75%; tail share of 99% ES 97.39%.


In [20]:
display(pd.DataFrame([stress['tail_impact']]).T.rename(columns={0: 'Tail share'}))
display(pd.DataFrame([stress['full_tail'], stress['tail_removed']], index=['Full EVT tail', 'Tail removed'])[['mean', 'var_95', 'expected_shortfall_95', 'var_99', 'expected_shortfall_99']])
display(pd.DataFrame(rigor['evt'])[['Quantile', 'ShapeMedian', 'ShapeLower', 'ShapeUpper', 'FiniteMeanProbability', 'FiniteVarianceProbability']])


,Tail share
mean_loss_share,0.627483
variance_share,1.000000
expected_shortfall_99_share,0.973858


,mean,var_95,expected_shortfall_95,var_99,expected_shortfall_99
Full EVT tail,1.051877e+08,1.410013e+08,4.539281e+08,3.132405e+08,1.525041e+09
Tail removed,3.918423e+07,3.960606e+07,3.971725e+07,3.979670e+07,3.986770e+07


,Quantile,ShapeMedian,ShapeLower,ShapeUpper,FiniteMeanProbability,FiniteVarianceProbability
0,0.900,0.844449,0.782652,0.919201,1.000000,0.0
1,0.950,0.914929,0.812995,1.011772,0.936667,0.0
2,0.975,1.021321,0.892821,1.171342,0.376667,0.0
3,0.990,0.840300,0.639293,1.066973,0.896667,0.0


## Overall conclusion

ML improves frequency estimation, severity MAE, and risk ranking, while actuarial models retain advantages in calibration, distributional fit, and transparent uncertainty. EVT better represents extreme losses, but its parameters are unstable and cannot compensate for weak policy-level large-loss predictability.

The most defensible architecture is hybrid: nonlinear frequency and ranking, a calibrated Tweedie benchmark for expected loss, interpretable component models for diagnosis, and separate EVT scenarios for tail stress testing.

Classifications: **Robust** has held-out and uncertainty support; **Suggestive** has consistent evidence with material trade-offs; **Exploratory** is scenario evidence with strong sensitivity; **Data-limited** means available predictors do not support a reliable operational conclusion.
